# Two-Stage TFLite Production Pipeline

## Architecture
- **Stage 1**: Binary classifier (Cancer vs Non-cancer) using MobileNetV3Large
- **Stage 2**: 4-class classifier (Benign, Healthy, OCA, OPMD) using MobileNetV3Large
- **Unknown**: Handled via confidence thresholding at inference time

## Baseline to Beat
- **TFLite 5-class model**: 69.66% accuracy

## Training Setup
Matches `tensorflow_pytorch_success_replication.ipynb` exactly:
- MobileNetV3Large backbone (pretrained ImageNet)
- Adam optimizer, lr=1e-4
- Batch size 32
- Early stopping patience=5
- Same data augmentation

In [1]:
# Cell 1: Config and Setup (M1-optimized TensorFlow)

import os
import json
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
from tqdm import tqdm

# M1 MacBook Pro optimizations - DISABLE XLA to avoid compilation errors
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.config.optimizer.set_jit(False)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(gpus) > 0}")
if gpus:
    print(f"GPU devices: {gpus}")

# Configuration - matches tensorflow_pytorch_success_replication.ipynb
CONFIG = {
    # Data paths
    'data_dir_binary': os.path.expanduser('~/dev/oral_cavity_binary_dataset'),
    'data_dir_5class': os.path.expanduser('~/dev/oral_cavity_5class_dataset'),
    'ensemble_test_dir': os.path.expanduser('~/dev/oral_cavity_5class_dataset/test'),
    
    # Model config - EXACT match to best TFLite model
    'image_size': (224, 224),
    'batch_size': 32,
    'learning_rate': 1e-4,
    'epochs': 30,
    'early_stopping_patience': 5,
    'dropout_rate': 0.2,
    
    # Class names
    'class_names_5class': ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown'],
    'class_names_4class': ['Benign', 'Healthy', 'OCA', 'OPMD'],
    'class_names_binary': ['non-cancer', 'cancer'],
    
    # Inference thresholds (unknown_threshold: max 4-class prob below this -> Unknown; 0.30 works for 4-class softmax)
    'cancer_threshold': 0.5,
    'unknown_threshold': 0.30,
}

# Output directory
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path(f'results/two_stage_tflite/run_{timestamp}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'models').mkdir(exist_ok=True)
(OUTPUT_DIR / 'tflite').mkdir(exist_ok=True)
(OUTPUT_DIR / 'plots').mkdir(exist_ok=True)

# Save config
with open(OUTPUT_DIR / 'config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"\nConfiguration:")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Image size: {CONFIG['image_size']}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  Epochs: {CONFIG['epochs']}")
print(f"  Early stopping patience: {CONFIG['early_stopping_patience']}")

TensorFlow version: 2.15.0
GPU available: True
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Configuration:
  Output directory: results/two_stage_tflite/run_20260129_103252
  Image size: (224, 224)
  Batch size: 32
  Learning rate: 0.0001
  Epochs: 30
  Early stopping patience: 5


In [2]:
# Cell 2: Data Loading with PyTorch-Style Augmentation

# Training augmentation - EXACT match to best TFLite model
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest'
)

# Validation/Test - no augmentation
val_test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)

print("Loading datasets...")
print("=" * 60)

# --- BINARY DATASET ---
print("\n[1] Binary Dataset (Cancer vs Non-cancer)")
binary_dir = Path(CONFIG['data_dir_binary'])
if binary_dir.exists():
    train_ds_binary = train_datagen.flow_from_directory(
        binary_dir,
        target_size=CONFIG['image_size'],
        batch_size=CONFIG['batch_size'],
        class_mode='binary',
        shuffle=True,
        seed=SEED
    )
    print(f"    Training samples: {train_ds_binary.samples}")
    print(f"    Classes: {train_ds_binary.class_indices}")
else:
    print(f"    ERROR: Directory not found: {binary_dir}")
    train_ds_binary = None

# --- 5-CLASS DATASET (for 4-class training, excluding Unknown) ---
print("\n[2] 5-Class Dataset (will filter to 4-class)")
data_dir_5class = Path(CONFIG['data_dir_5class'])

if data_dir_5class.exists():
    # Load all 5 classes
    train_ds_5class = train_datagen.flow_from_directory(
        data_dir_5class / 'train',
        target_size=CONFIG['image_size'],
        batch_size=CONFIG['batch_size'],
        class_mode='sparse',
        shuffle=True,
        seed=SEED
    )
    
    val_ds_5class = val_test_datagen.flow_from_directory(
        data_dir_5class / 'val',
        target_size=CONFIG['image_size'],
        batch_size=CONFIG['batch_size'],
        class_mode='sparse',
        shuffle=False,
        seed=SEED
    )
    
    test_ds_5class = val_test_datagen.flow_from_directory(
        data_dir_5class / 'test',
        target_size=CONFIG['image_size'],
        batch_size=CONFIG['batch_size'],
        class_mode='sparse',
        shuffle=False,
        seed=SEED
    )
    
    print(f"    Training: {train_ds_5class.samples} samples")
    print(f"    Validation: {val_ds_5class.samples} samples")
    print(f"    Test: {test_ds_5class.samples} samples")
    print(f"    Classes: {list(train_ds_5class.class_indices.keys())}")
else:
    print(f"    ERROR: Directory not found: {data_dir_5class}")
    train_ds_5class = val_ds_5class = test_ds_5class = None

# --- Create 4-class datasets (excluding Unknown) ---
print("\n[3] Creating 4-class datasets (excluding Unknown)...")

def create_4class_generator(base_dir, split, datagen, batch_size, shuffle=True):
    """Create a generator that only includes 4 classes (no Unknown)"""
    class_names_4 = ['Benign', 'Healthy', 'OCA', 'OPMD']
    
    # Collect image paths and labels
    images = []
    labels = []
    
    split_dir = Path(base_dir) / split
    for idx, class_name in enumerate(class_names_4):
        class_dir = split_dir / class_name
        if class_dir.exists():
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
                for img_path in class_dir.glob(ext):
                    images.append(str(img_path))
                    labels.append(idx)
    
    print(f"    {split}: {len(images)} samples (4-class)")
    
    # Create tf.data.Dataset
    def load_and_preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, CONFIG['image_size'])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label
    
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    
    return ds, len(images)

train_ds_4class, train_count_4class = create_4class_generator(
    CONFIG['data_dir_5class'], 'train', train_datagen, CONFIG['batch_size'], shuffle=True
)
val_ds_4class, val_count_4class = create_4class_generator(
    CONFIG['data_dir_5class'], 'val', val_test_datagen, CONFIG['batch_size'], shuffle=False
)
test_ds_4class, test_count_4class = create_4class_generator(
    CONFIG['data_dir_5class'], 'test', val_test_datagen, CONFIG['batch_size'], shuffle=False
)

# --- Create binary validation set from 5-class data ---
print("\n[4] Creating binary validation set from 5-class data...")

def create_binary_from_5class(base_dir, split, batch_size, shuffle=False):
    """Map 5-class to binary: OCA/OPMD -> cancer (1), Benign/Healthy/Unknown -> non-cancer (0)"""
    class_names_5 = ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown']
    
    images = []
    labels = []  # 0 = non-cancer, 1 = cancer
    
    split_dir = Path(base_dir) / split
    for class_name in class_names_5:
        class_dir = split_dir / class_name
        is_cancer = 1 if class_name in ['OCA', 'OPMD'] else 0
        if class_dir.exists():
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
                for img_path in class_dir.glob(ext):
                    images.append(str(img_path))
                    labels.append(is_cancer)
    
    print(f"    {split}: {len(images)} samples (binary from 5-class)")
    cancer_count = sum(labels)
    print(f"        Cancer: {cancer_count}, Non-cancer: {len(labels) - cancer_count}")
    
    def load_and_preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, CONFIG['image_size'])
        img = tf.cast(img, tf.float32) / 255.0
        return img, tf.cast(label, tf.float32)
    
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    
    return ds, len(images)

val_ds_binary_from_5class, _ = create_binary_from_5class(
    CONFIG['data_dir_5class'], 'val', CONFIG['batch_size'], shuffle=False
)

print("\nDatasets ready!")

Loading datasets...

[1] Binary Dataset (Cancer vs Non-cancer)
Found 1634 images belonging to 2 classes.
    Training samples: 1634
    Classes: {'cancer': 0, 'non-cancer': 1}

[2] 5-Class Dataset (will filter to 4-class)
Found 2757 images belonging to 5 classes.
Found 531 images belonging to 5 classes.
Found 534 images belonging to 5 classes.
    Training: 2757 samples
    Validation: 531 samples
    Test: 534 samples
    Classes: ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown']

[3] Creating 4-class datasets (excluding Unknown)...
    train: 2357 samples (4-class)
    val: 481 samples (4-class)
    test: 484 samples (4-class)

[4] Creating binary validation set from 5-class data...
    val: 531 samples (binary from 5-class)
        Cancer: 244, Non-cancer: 287

Datasets ready!


In [3]:
# Cell 3: Model Architecture - MobileNetV3Large (matches best TFLite model)

def build_mobilenetv3_model(num_classes, input_shape=(224, 224, 3), dropout_rate=0.2):
    """
    Build MobileNetV3Large model matching the best TFLite model architecture exactly.
    
    Architecture from tensorflow_pytorch_success_replication.ipynb:
    - MobileNetV3Large backbone (pretrained ImageNet)
    - GlobalAveragePooling2D
    - Dense(1280, relu)
    - Dropout(0.2)
    - Dense(num_classes, softmax/sigmoid)
    """
    # Load pre-trained MobileNetV3Large
    base_model = MobileNetV3Large(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
        alpha=1.0,
        minimalistic=False
    )
    
    # Build model
    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    
    # Global average pooling
    x = layers.GlobalAveragePooling2D()(x)
    
    # Custom classifier head (matching best TFLite model)
    x = layers.Dense(1280, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    
    # Output layer
    if num_classes == 1:
        # Binary classification
        outputs = layers.Dense(1, activation='sigmoid')(x)
    else:
        # Multi-class classification
        outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    return model

print("Building models...")
print("=" * 60)

# Binary model (Cancer vs Non-cancer)
print("\n[1] Binary Model (Cancer vs Non-cancer)")
binary_model = build_mobilenetv3_model(
    num_classes=1,
    input_shape=(*CONFIG['image_size'], 3),
    dropout_rate=CONFIG['dropout_rate']
)
binary_model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=CONFIG['learning_rate']),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(f"    Parameters: {binary_model.count_params():,}")
print(f"    Size estimate: {binary_model.count_params() * 4 / (1024*1024):.1f} MB")

# 4-class model (Benign, Healthy, OCA, OPMD)
print("\n[2] 4-Class Model (Benign, Healthy, OCA, OPMD)")
four_class_model = build_mobilenetv3_model(
    num_classes=4,
    input_shape=(*CONFIG['image_size'], 3),
    dropout_rate=CONFIG['dropout_rate']
)
four_class_model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=CONFIG['learning_rate']),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print(f"    Parameters: {four_class_model.count_params():,}")
print(f"    Size estimate: {four_class_model.count_params() * 4 / (1024*1024):.1f} MB")

print("\nModels built successfully!")

Building models...

[1] Binary Model (Cancer vs Non-cancer)
    Parameters: 4,227,713
    Size estimate: 16.1 MB

[2] 4-Class Model (Benign, Healthy, OCA, OPMD)
    Parameters: 4,231,556
    Size estimate: 16.1 MB

Models built successfully!


In [4]:
# Cell 4: Training - Binary Model

print("Training Binary Model (Cancer vs Non-cancer)")
print("=" * 60)

# Callbacks
binary_model_path = str(OUTPUT_DIR / 'models' / 'binary_model.keras')
binary_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['early_stopping_patience'],
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        binary_model_path,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

# Train binary model
if train_ds_binary is not None:
    binary_history = binary_model.fit(
        train_ds_binary,
        validation_data=val_ds_binary_from_5class,
        epochs=CONFIG['epochs'],
        callbacks=binary_callbacks,
        verbose=1
    )
    
    # Save training history
    with open(OUTPUT_DIR / 'binary_training_history.json', 'w') as f:
        json.dump({k: [float(v) for v in vals] for k, vals in binary_history.history.items()}, f, indent=2)
    
    print(f"\nBinary model training complete!")
    print(f"Best val_loss: {min(binary_history.history['val_loss']):.4f}")
    print(f"Best val_accuracy: {max(binary_history.history['val_accuracy']):.4f}")
else:
    print("ERROR: Binary training dataset not available!")

Training Binary Model (Cancer vs Non-cancer)
Epoch 1/30
 2/52 [>.............................] - ETA: 10s - loss: 0.8435 - accuracy: 0.5938  

/Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/venv_multihead_tflite/lib/python3.11/site-packages/PIL/Image.py:970: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


52/52 [==============================] - ETA: 0s - loss: 0.7129 - accuracy: 0.5349
Epoch 1: val_loss improved from inf to 0.69577, saving model to results/two_stage_tflite/run_20260129_103252/models/binary_model.keras
52/52 [==============================] - 41s 552ms/step - loss: 0.7129 - accuracy: 0.5349 - val_loss: 0.6958 - val_accuracy: 0.4595 - lr: 1.0000e-04
Epoch 2/30
52/52 [==============================] - ETA: 0s - loss: 0.7001 - accuracy: 0.5086
Epoch 2: val_loss did not improve from 0.69577
52/52 [==============================] - 23s 432ms/step - loss: 0.7001 - accuracy: 0.5086 - val_loss: 0.7032 - val_accuracy: 0.5405 - lr: 1.0000e-04
Epoch 3/30
52/52 [==============================] - ETA: 0s - loss: 0.6919 - accuracy: 0.5226
Epoch 3: val_loss improved from 0.69577 to 0.68980, saving model to results/two_stage_tflite/run_20260129_103252/models/binary_model.keras
52/52 [==============================] - 24s 448ms/step - loss: 0.6919 - accuracy: 0.5226 - val_loss: 0.6898 -

In [5]:
# Cell 5: Training - 4-Class Model

print("Training 4-Class Model (Benign, Healthy, OCA, OPMD)")
print("=" * 60)

# Callbacks
four_class_model_path = str(OUTPUT_DIR / 'models' / 'four_class_model.keras')
four_class_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['early_stopping_patience'],
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        four_class_model_path,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

# Calculate steps per epoch
steps_per_epoch_4class = train_count_4class // CONFIG['batch_size']
validation_steps_4class = val_count_4class // CONFIG['batch_size']

print(f"Steps per epoch: {steps_per_epoch_4class}")
print(f"Validation steps: {validation_steps_4class}")

# Train 4-class model
four_class_history = four_class_model.fit(
    train_ds_4class,
    validation_data=val_ds_4class,
    epochs=CONFIG['epochs'],
    steps_per_epoch=steps_per_epoch_4class,
    validation_steps=validation_steps_4class,
    callbacks=four_class_callbacks,
    verbose=1
)

# Save training history
with open(OUTPUT_DIR / 'four_class_training_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in four_class_history.history.items()}, f, indent=2)

print(f"\n4-Class model training complete!")
print(f"Best val_loss: {min(four_class_history.history['val_loss']):.4f}")
print(f"Best val_accuracy: {max(four_class_history.history['val_accuracy']):.4f}")

Training 4-Class Model (Benign, Healthy, OCA, OPMD)
Steps per epoch: 73
Validation steps: 15
Epoch 1/30
73/73 [==============================] - ETA: 0s - loss: 1.3040 - accuracy: 0.3926
Epoch 1: val_loss improved from inf to 1.24744, saving model to results/two_stage_tflite/run_20260129_103252/models/four_class_model.keras
73/73 [==============================] - 33s 366ms/step - loss: 1.3040 - accuracy: 0.3926 - val_loss: 1.2474 - val_accuracy: 0.4333 - lr: 1.0000e-04
Epoch 2/30
 1/73 [..............................] - ETA: 2:44 - loss: 1.3561 - accuracy: 0.2857WARNING:tensorflow:Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches (in this case, 2190 batches). You may need to use the repeat() function when building your dataset.

Epoch 2: val_loss did not improve from 1.24744
73/73 [==============================] - 7s 67ms/step - loss: 1.3561 - accuracy: 0.2857 - val_loss: 1.2475 - val_a

In [6]:
# Cell 6: TFLite Conversion

print("Converting models to TFLite...")
print("=" * 60)

def convert_to_tflite(model, output_path, quantization='dynamic'):
    """
    Convert Keras model to TFLite with specified quantization.
    
    Args:
        model: Keras model to convert
        output_path: Path to save TFLite model
        quantization: 'none', 'dynamic', or 'float16'
    """
    print(f"\nConverting to TFLite ({quantization} quantization)...")
    
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    if quantization == 'dynamic':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif quantization == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    # 'none' = no optimization
    
    try:
        tflite_model = converter.convert()
        
        with open(output_path, 'wb') as f:
            f.write(tflite_model)
        
        size_mb = len(tflite_model) / (1024 * 1024)
        print(f"    Saved: {output_path}")
        print(f"    Size: {size_mb:.2f} MB")
        return output_path, size_mb
    except Exception as e:
        print(f"    ERROR: {e}")
        return None, 0

# Load best models
print("Loading best models...")
binary_model_best = keras.models.load_model(binary_model_path)
four_class_model_best = keras.models.load_model(four_class_model_path)

# Convert binary model
print("\n[1] Binary Model")
binary_tflite_path, binary_size = convert_to_tflite(
    binary_model_best,
    str(OUTPUT_DIR / 'tflite' / 'binary_model_dynamic.tflite'),
    quantization='dynamic'
)

# Convert 4-class model
print("\n[2] 4-Class Model")
four_class_tflite_path, four_class_size = convert_to_tflite(
    four_class_model_best,
    str(OUTPUT_DIR / 'tflite' / 'four_class_model_dynamic.tflite'),
    quantization='dynamic'
)

print(f"\nTotal TFLite size: {binary_size + four_class_size:.2f} MB")
print("TFLite conversion complete!")

Converting models to TFLite...
Loading best models...

[1] Binary Model

Converting to TFLite (dynamic quantization)...
INFO:tensorflow:Assets written to: /var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/tmp0r6hqtqj/assets


INFO:tensorflow:Assets written to: /var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/tmp0r6hqtqj/assets
Summary on the non-converted ops:
---------------------------------
 * Accepted dialects: tfl, builtin, func
 * Non-Converted Ops: 75, Total Ops 263, % non-converted = 28.52 %
 * 75 ARITH ops

- arith.constant:   75 occurrences  (f32: 72, i32: 3)



  (f32: 11)
  (f32: 47)
  (f32: 15)
  (f32: 2)
  (f32: 20)
  (f32: 1)
  (f32: 9)
  (f32: 17)
  (f32: 4)
  (uq_8: 59)


    Saved: results/two_stage_tflite/run_20260129_103252/tflite/binary_model_dynamic.tflite
    Size: 4.31 MB

[2] 4-Class Model

Converting to TFLite (dynamic quantization)...
INFO:tensorflow:Assets written to: /var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/tmpqr4i2ha_/assets


INFO:tensorflow:Assets written to: /var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/tmpqr4i2ha_/assets


    Saved: results/two_stage_tflite/run_20260129_103252/tflite/four_class_model_dynamic.tflite
    Size: 4.32 MB

Total TFLite size: 8.63 MB
TFLite conversion complete!


Summary on the non-converted ops:
---------------------------------
 * Accepted dialects: tfl, builtin, func
 * Non-Converted Ops: 75, Total Ops 263, % non-converted = 28.52 %
 * 75 ARITH ops

- arith.constant:   75 occurrences  (f32: 72, i32: 3)



  (f32: 11)
  (f32: 47)
  (f32: 15)
  (f32: 2)
  (f32: 20)
  (f32: 9)
  (f32: 17)
  (f32: 4)
  (uq_8: 59)
  (f32: 1)


In [7]:
# Cell 7: Two-Stage Inference Logic

print("Setting up two-stage inference...")
print("=" * 60)

class TwoStageClassifier:
    """
    Two-stage oral cavity classifier.
    
    Stage 1: Binary model predicts cancer vs non-cancer
    Stage 2: 4-class model predicts Benign/Healthy/OCA/OPMD
    Unknown: Emitted when confidence is below threshold or models disagree
    """
    
    def __init__(self, binary_model, four_class_model, 
                 cancer_threshold=0.5, unknown_threshold=0.55):
        self.binary_model = binary_model
        self.four_class_model = four_class_model
        self.cancer_threshold = cancer_threshold
        self.unknown_threshold = unknown_threshold
        self.class_names = ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown']
    
    def predict_single(self, image):
        """
        Predict class for a single image.
        
        Args:
            image: Preprocessed image (224, 224, 3), normalized to [0, 1]
        
        Returns:
            dict with prediction details
        """
        # Ensure batch dimension
        if len(image.shape) == 3:
            image = np.expand_dims(image, axis=0)
        
        # Stage 1: Binary classification
        cancer_prob = float(self.binary_model.predict(image, verbose=0)[0][0])
        
        # Stage 2: 4-class classification
        four_probs = self.four_class_model.predict(image, verbose=0)[0]
        top_idx = int(np.argmax(four_probs))
        top_conf = float(np.max(four_probs))
        
        # Decision logic
        if top_conf < self.unknown_threshold:
            final_class = 4  # Unknown (low confidence)
            reason = 'low_confidence'
        elif cancer_prob >= self.cancer_threshold:
            # Binary says cancer -> only allow OCA/OPMD
            if top_idx in [2, 3]:  # OCA or OPMD
                final_class = top_idx
                reason = 'cancer_match'
            else:
                # 4-class says Benign/Healthy but binary says cancer
                # Conservative fallback to OPMD (high-risk)
                final_class = 3  # OPMD
                reason = 'cancer_override'
        else:
            # Binary says non-cancer -> only allow Benign/Healthy
            if top_idx in [0, 1]:  # Benign or Healthy
                final_class = top_idx
                reason = 'non_cancer_match'
            else:
                # 4-class says OCA/OPMD but binary says non-cancer
                # Emit Unknown on conflict
                final_class = 4  # Unknown
                reason = 'conflict'
        
        return {
            'final_class': final_class,
            'final_class_name': self.class_names[final_class],
            'cancer_prob': cancer_prob,
            'four_class_probs': four_probs.tolist(),
            'four_class_top': top_idx,
            'four_class_conf': top_conf,
            'reason': reason
        }
    
    def predict_batch(self, images):
        """
        Predict classes for a batch of images.
        
        Args:
            images: Batch of preprocessed images (N, 224, 224, 3)
        
        Returns:
            list of prediction dicts
        """
        # Stage 1: Binary classification
        cancer_probs = self.binary_model.predict(images, verbose=0).flatten()
        
        # Stage 2: 4-class classification
        four_probs_batch = self.four_class_model.predict(images, verbose=0)
        
        results = []
        for i in range(len(images)):
            cancer_prob = float(cancer_probs[i])
            four_probs = four_probs_batch[i]
            top_idx = int(np.argmax(four_probs))
            top_conf = float(np.max(four_probs))
            
            # Decision logic (same as predict_single)
            if top_conf < self.unknown_threshold:
                final_class = 4
                reason = 'low_confidence'
            elif cancer_prob >= self.cancer_threshold:
                if top_idx in [2, 3]:
                    final_class = top_idx
                    reason = 'cancer_match'
                else:
                    final_class = 3
                    reason = 'cancer_override'
            else:
                if top_idx in [0, 1]:
                    final_class = top_idx
                    reason = 'non_cancer_match'
                else:
                    final_class = 4
                    reason = 'conflict'
            
            results.append({
                'final_class': final_class,
                'cancer_prob': cancer_prob,
                'four_class_conf': top_conf,
                'reason': reason
            })
        
        return results

# Create classifier instance
classifier = TwoStageClassifier(
    binary_model=binary_model_best,
    four_class_model=four_class_model_best,
    cancer_threshold=CONFIG['cancer_threshold'],
    unknown_threshold=CONFIG['unknown_threshold']
)

print(f"Two-stage classifier ready!")
print(f"  Cancer threshold: {CONFIG['cancer_threshold']}")
print(f"  Unknown threshold: {CONFIG['unknown_threshold']}")

Setting up two-stage inference...
Two-stage classifier ready!
  Cancer threshold: 0.5
  Unknown threshold: 0.55


In [8]:
# Cell 8: Evaluation - Apples-to-Apples Comparison with TFLite Baseline

print("Evaluating Two-Stage Pipeline on 5-Class Test Set")
print("=" * 60)
print(f"Baseline to beat: TFLite 5-class model = 69.66% accuracy")
print()

# Load test images
test_dir = Path(CONFIG['ensemble_test_dir'])
if not test_dir.exists():
    print(f"ERROR: Test directory not found: {test_dir}")
else:
    # Collect all test images with labels
    test_images = []
    test_labels = []
    
    for class_idx, class_name in enumerate(CONFIG['class_names_5class']):
        class_dir = test_dir / class_name
        if class_dir.exists():
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
                for img_path in class_dir.glob(ext):
                    test_images.append(str(img_path))
                    test_labels.append(class_idx)
    
    print(f"Test samples: {len(test_images)}")
    for i, name in enumerate(CONFIG['class_names_5class']):
        count = sum(1 for l in test_labels if l == i)
        print(f"  {name}: {count}")
    
    # Evaluate in batches (collect full results for confidence diagnostics)
    all_preds = []
    all_results = []  # full result dicts with four_class_conf, reason, etc.
    all_labels = test_labels
    batch_size = CONFIG['batch_size']
    
    print(f"\nRunning inference...")
    for i in tqdm(range(0, len(test_images), batch_size), desc='Evaluating'):
        batch_paths = test_images[i:i+batch_size]
        
        # Load and preprocess batch (must match training: resize, /255)
        batch_imgs = []
        for path in batch_paths:
            img = tf.io.read_file(path)
            img = tf.image.decode_jpeg(img, channels=3)
            img = tf.image.resize(img, CONFIG['image_size'])
            img = tf.cast(img, tf.float32) / 255.0
            batch_imgs.append(img.numpy())
        
        batch_imgs = np.array(batch_imgs)
        
        # Get predictions
        results = classifier.predict_batch(batch_imgs)
        all_preds.extend([r['final_class'] for r in results])
        all_results.extend(results)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Diagnostic: 4-class confidence (max softmax prob) - explains Unknown rate
    four_class_confs = np.array([r['four_class_conf'] for r in all_results])
    reason_counts = {}
    for r in all_results:
        reason_counts[r['reason']] = reason_counts.get(r['reason'], 0) + 1
    print(f"\n4-Class confidence (max prob): min={four_class_confs.min():.3f}, max={four_class_confs.max():.3f}, mean={four_class_confs.mean():.3f}")
    print(f"  Samples with max prob < 0.55: {(four_class_confs < 0.55).sum()} ({(four_class_confs < 0.55).mean()*100:.1f}%) -> would be Unknown if unknown_threshold=0.55")
    print(f"  Samples with max prob < 0.30: {(four_class_confs < 0.30).sum()} ({(four_class_confs < 0.30).mean()*100:.1f}%) -> Unknown with current threshold=0.30")
    print(f"  Prediction reasons: {reason_counts}")
    
    # Calculate metrics
    overall_accuracy = float(np.mean(all_preds == all_labels))
    
    print(f"\n" + "=" * 60)
    print(f"RESULTS: Two-Stage TFLite Pipeline")
    print(f"=" * 60)
    print(f"\nOverall Accuracy: {overall_accuracy*100:.2f}%")
    print(f"Baseline (TFLite 5-class): 69.66%")
    print(f"Difference: {(overall_accuracy*100 - 69.66):+.2f}%")
    
    if overall_accuracy > 0.6966:
        print(f"\n>>> SUCCESS: Two-stage pipeline beats TFLite baseline! <<<")
    else:
        print(f"\n>>> Two-stage pipeline did not beat baseline <<<")
    
    # Per-class accuracy
    print(f"\nPer-Class Accuracy:")
    for i, name in enumerate(CONFIG['class_names_5class']):
        mask = all_labels == i
        if mask.sum() > 0:
            class_acc = float(np.mean(all_preds[mask] == all_labels[mask]))
            print(f"  {name}: {class_acc*100:.2f}% ({mask.sum()} samples)")
        else:
            print(f"  {name}: N/A (0 samples)")
    
    # Confusion matrix
    print(f"\nConfusion Matrix:")
    from sklearn.metrics import confusion_matrix, classification_report
    cm = confusion_matrix(all_labels, all_preds, labels=range(5))
    print(f"             Predicted")
    print(f"             " + " ".join([f"{n[:4]:>6}" for n in CONFIG['class_names_5class']]))
    for i, name in enumerate(CONFIG['class_names_5class']):
        row = " ".join([f"{cm[i,j]:>6}" for j in range(5)])
        print(f"  {name:>8}: {row}")
    
    # Prediction distribution
    print(f"\nPrediction Distribution:")
    unique, counts = np.unique(all_preds, return_counts=True)
    for u, c in zip(unique, counts):
        pct = c / len(all_preds) * 100
        print(f"  {CONFIG['class_names_5class'][u]}: {c} ({pct:.1f}%)")
    
    # Save results
    results_summary = {
        'overall_accuracy': overall_accuracy,
        'baseline_accuracy': 0.6966,
        'improvement': overall_accuracy - 0.6966,
        'total_samples': len(all_labels),
        'per_class_accuracy': {},
        'confusion_matrix': cm.tolist()
    }
    for i, name in enumerate(CONFIG['class_names_5class']):
        mask = all_labels == i
        if mask.sum() > 0:
            results_summary['per_class_accuracy'][name] = float(np.mean(all_preds[mask] == all_labels[mask]))
    
    with open(OUTPUT_DIR / 'evaluation_results.json', 'w') as f:
        json.dump(results_summary, f, indent=2)
    
    print(f"\nResults saved to: {OUTPUT_DIR / 'evaluation_results.json'}")

Evaluating Two-Stage Pipeline on 5-Class Test Set
Baseline to beat: TFLite 5-class model = 69.66% accuracy

Test samples: 534
  Benign: 129
  Healthy: 110
  OCA: 36
  OPMD: 209
  Unknown: 50

Running inference...


Evaluating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 17/17 [00:42<00:00,  2.49s/it]



RESULTS: Two-Stage TFLite Pipeline

Overall Accuracy: 9.36%
Baseline (TFLite 5-class): 69.66%
Difference: -60.30%

>>> Two-stage pipeline did not beat baseline <<<

Per-Class Accuracy:
  Benign: 0.00% (129 samples)
  Healthy: 0.00% (110 samples)
  OCA: 0.00% (36 samples)
  OPMD: 0.00% (209 samples)
  Unknown: 100.00% (50 samples)

Confusion Matrix:
             Predicted
               Beni   Heal    OCA   OPMD   Unkn
    Benign:      0      0      0      0    129
   Healthy:      0      0      0      0    110
       OCA:      0      0      0      0     36
      OPMD:      0      0      0      0    209
   Unknown:      0      0      0      0     50

Prediction Distribution:
  Unknown: 534 (100.0%)

Results saved to: results/two_stage_tflite/run_20260129_103252/evaluation_results.json


In [ ]:
# Cell 9: Summary and Deployment Instructions

print("=" * 60)
print("TWO-STAGE TFLITE PIPELINE - SUMMARY")
print("=" * 60)

print(f"\nOutput Directory: {OUTPUT_DIR}")

print(f"\nModels:")
print(f"  [1] Binary Model (Cancer vs Non-cancer)")
print(f"      Keras: {OUTPUT_DIR / 'models' / 'binary_model.keras'}")
print(f"      TFLite: {OUTPUT_DIR / 'tflite' / 'binary_model_dynamic.tflite'}")
if binary_tflite_path:
    print(f"      Size: {binary_size:.2f} MB")

print(f"\n  [2] 4-Class Model (Benign, Healthy, OCA, OPMD)")
print(f"      Keras: {OUTPUT_DIR / 'models' / 'four_class_model.keras'}")
print(f"      TFLite: {OUTPUT_DIR / 'tflite' / 'four_class_model_dynamic.tflite'}")
if four_class_tflite_path:
    print(f"      Size: {four_class_size:.2f} MB")

print(f"\nTotal TFLite Size: {binary_size + four_class_size:.2f} MB")

print(f"\nInference Thresholds:")
print(f"  Cancer threshold: {CONFIG['cancer_threshold']}")
print(f"  Unknown threshold: {CONFIG['unknown_threshold']}")

print(f"\n" + "=" * 60)
print("DEPLOYMENT INSTRUCTIONS")
print("=" * 60)

print(f"""
1. Copy TFLite models to Flutter assets:
   cp {OUTPUT_DIR / 'tflite' / 'binary_model_dynamic.tflite'} ../../assets/
   cp {OUTPUT_DIR / 'tflite' / 'four_class_model_dynamic.tflite'} ../../assets/

2. Update Flutter model_inference_service.dart:
   - Load both TFLite models
   - Implement two-stage inference logic
   - Use cancer_threshold={CONFIG['cancer_threshold']}, unknown_threshold={CONFIG['unknown_threshold']}

3. Inference Flow:
   a) Run binary model: cancer_prob = binary_model(image)
   b) Run 4-class model: [benign, healthy, oca, opmd] = four_class_model(image)
   c) Apply decision logic:
      - If max(4-class) < {CONFIG['unknown_threshold']}: return Unknown
      - If cancer_prob >= {CONFIG['cancer_threshold']}:
          - If 4-class says OCA/OPMD: return that class
          - Else: return OPMD (conservative)
      - Else (non-cancer):
          - If 4-class says Benign/Healthy: return that class
          - Else: return Unknown (conflict)
""")

print("\nTraining complete!")